# Trace 01: multimer SASA (`SASAMultimer`)

Runs `dsa.sasa.sasa_multimer.SASAMultimer` from scratch on structures with an increasing number of chains in the
biological assembly, from 2 up to the largest assembly in the data, and checks every result against the saved copy in
`SASAMultimerStore`.

**Nothing is written:** creating `SASAMultimer(structure_id)` computes in memory; only `ComputeManager` saves to the store.

Each structure is an X-ray structure (except the largest) that contains lysines measured in the MS data, chosen as the smallest one for its chain count so it runs quickly:

| chains | structure | why |
|---|---|---|
| 2 | `4ah6` | homodimer of Q6PI48; K224 has an inter-chain salt bridge |
| 3 | `3r45` | heteromer HJURP (Q8NCD3) + CENP-A + histone H4; K39 inter-chain salt bridge |
| 4 | `7sul` | O94979 homotetramer; NZ missing in most copies of K200 and K206 |
| 5 | `1w9e` | homopentamer, 1.56 Å |
| 6 | `2jf1` | heteromer of 2 UniProt ids |
| 7 | `4wzj` | 7 chains, 7 different UniProt ids |
| 8 | `4z9v` | 2 UniProt ids, 8 measured lysines |
| 9 | `9azj` | 3 UniProt ids, 12 measured lysines |
| 10 | `9ubd` | 2 UniProt ids, 14 measured lysines |
| 24 | `8cn1` | homo 24-mer; one measured lysine per copy |
| 974 | `8j07` | largest assembly in the data (cryo-EM, 103 UniProt ids, 396,307 residues); about 3 min and 5.5 GB, off by default |

Change `STRUCTURES` to trace other structures.

In [1]:
import time
import warnings
import pandas as pd
from dsa.sasa.sasa_multimer import SASAMultimer, SASAMultimerStore

# the UniProt aligner warns once per structure residue that is not in the PDB-UniProt alignment
# (e.g. expression tags); those residues get uniprot_id = None, counted as unmapped_residues below
warnings.filterwarnings("ignore", message="Invalid residue id")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
store = SASAMultimerStore()

## Parameters

In [2]:
STRUCTURES = {2: "4ah6", 3: "3r45", 4: "7sul", 5: "1w9e", 6: "2jf1", 7: "4wzj",
              8: "4z9v", 9: "9azj", 10: "9ubd", 24: "8cn1"}
LARGEST = "8j07"
RUN_LARGEST = False   # about 3 minutes and 5.5 GB of memory

## Helpers

`run(structure_id)` computes the SASA, times it and compares it with the stored result.
`chain_table(obj)` summarises the result per chain copy in the biological assembly.

In [3]:
def run(structure_id: str) -> tuple[SASAMultimer, dict]:
    start = time.time()
    obj = SASAMultimer(structure_id)
    seconds = time.time() - start
    df = obj.sasa_df
    stored = store.load(structure_id)
    try:
        pd.testing.assert_frame_equal(df, stored.sasa_df)
        matches_store = True
    except AssertionError as e:
        matches_store = False
        print(f"{structure_id}: differs from the stored result:\n{e}")
    lysines = df[df["resname3"] == "LYS"]
    info = {"structure_id": structure_id,
            "bio_chains": df["bio_chain"].nunique(),
            "author_chains": df["author_chain"].nunique(),
            "uniprot_ids": len(obj.uniprot_ids),
            "residues": len(df),
            "lysines": len(lysines),
            "lysines_with_nz": int(lysines["asa_nz"].notna().sum()),
            "unmapped_residues": int(df["uniprot_id"].isna().sum()),
            "seconds": round(seconds, 1),
            "matches_store": matches_store}
    return obj, info


def chain_table(obj: SASAMultimer) -> pd.DataFrame:
    df = obj.sasa_df
    return (df.assign(is_lys=df["resname3"] == "LYS", nz=df["asa_nz"].notna())
              .groupby(["bio_chain", "author_chain"], dropna=False)
              .agg(uniprot_id=("uniprot_id", "first"), residues=("resnum", "size"),
                   lysines=("is_lys", "sum"), lysines_with_nz=("nz", "sum"),
                   asa_total=("asa", "sum"), asa_nz_mean=("asa_nz", "mean"))
              .round(2).reset_index())

## Run all structures

In [4]:
results, infos = {}, []
for n_chains, structure_id in STRUCTURES.items():
    obj, info = run(structure_id)
    results[structure_id] = obj
    infos.append({"expected_chains": n_chains, **info})
summary = pd.DataFrame(infos)
summary

,expected_chains,structure_id,bio_chains,author_chains,uniprot_ids,residues,lysines,lysines_with_nz,unmapped_residues,seconds,matches_store
0,2,4ah6,2,2,1,1178,64,64,0,0.9,True
1,3,3r45,3,3,3,217,12,12,0,0.2,True
2,4,7sul,4,4,1,1330,54,23,8,0.8,True
3,5,1w9e,5,5,1,342,24,24,20,0.3,True
4,6,2jf1,6,2,2,294,12,12,0,0.3,True
5,7,4wzj,7,7,7,575,49,48,0,1.0,True
6,8,4z9v,8,8,2,418,16,16,1,0.4,True
7,9,9azj,9,9,3,589,57,30,6,0.4,True
8,10,9ubd,10,5,2,730,48,30,20,0.5,True
9,24,8cn1,24,24,1,1190,59,52,87,0.8,True


In [5]:
assert (summary["bio_chains"] == summary["expected_chains"]).all(), "unexpected number of chains"
assert summary["matches_store"].all(), "a result differs from the stored one"
print("All structures match the stored results.")

All structures match the stored results.


## Inspect one structure

Per chain copy, then the lysines. `asa_nz` is NaN when the lysine's NZ atom is not in the deposited model
(truncated side chain); `asa` of the residue is still computed from the atoms that are present.

In [6]:
structure_id = "4ah6"
obj = results[structure_id]
print(obj.summary())
chain_table(obj)

4ah6: 2 chains, 1178 residues, 64 lysine NZ, uniprot ids ['Q6PI48']


,bio_chain,author_chain,uniprot_id,residues,lysines,lysines_with_nz,asa_total,asa_nz_mean
0,A1,A,Q6PI48,589,32,32,25559.82,36.21
1,B1,B,Q6PI48,589,32,32,25645.77,36.34


In [7]:
lysines = obj.sasa_df[obj.sasa_df["resname3"] == "LYS"]
lysines[["bio_chain", "author_chain", "resnum", "icode", "uniprot_id", "unp_resnum",
         "asa", "rel_asa", "asa_sidechain", "asa_nz"]].head(20)

,bio_chain,author_chain,resnum,icode,uniprot_id,unp_resnum,asa,rel_asa,asa_sidechain,asa_nz
62,A1,A,104,,Q6PI48,104,47.873,0.2336,47.747,26.617
63,A1,A,105,,Q6PI48,105,146.929,0.7168,139.467,57.452
91,A1,A,133,,Q6PI48,133,193.164,0.9424,152.748,51.387
100,A1,A,142,,Q6PI48,142,79.364,0.3872,77.775,23.063
102,A1,A,144,,Q6PI48,144,150.292,0.7332,133.542,52.778
111,A1,A,153,,Q6PI48,153,93.490,0.4561,83.004,18.498
112,A1,A,154,,Q6PI48,154,166.502,0.8123,155.223,52.789
118,A1,A,160,,Q6PI48,160,150.392,0.7337,135.876,52.387
122,A1,A,164,,Q6PI48,164,110.098,0.5371,95.600,32.843
123,A1,A,165,,Q6PI48,165,167.803,0.8186,131.033,41.926


### Same lysine in every chain copy

Copies of a chain share the UniProt numbering, so one UniProt lysine has one row per copy. Copies can differ
when they sit in different environments in the assembly.

In [8]:
uniprot_id, unp_resnum = "Q6PI48", 224
obj.sasa_df[(obj.sasa_df["uniprot_id"] == uniprot_id) & (obj.sasa_df["unp_resnum"] == unp_resnum)][
    ["bio_chain", "author_chain", "resnum", "resname3", "asa", "asa_sidechain", "asa_nz"]]

,bio_chain,author_chain,resnum,resname3,asa,asa_sidechain,asa_nz
182,A1,A,224,LYS,103.279,98.172,20.336
771,B1,B,224,LYS,102.376,97.929,20.224


## Largest assembly (optional)

In [9]:
if RUN_LARGEST:
    obj, info = run(LARGEST)
    results[LARGEST] = obj
    display(pd.DataFrame([info]))
    display(chain_table(obj).head(20))
else:
    print(f"Skipped {LARGEST}; set RUN_LARGEST = True to run it.")

Skipped 8j07; set RUN_LARGEST = True to run it.
